In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 2 — User Inputs
# Each variable is prompted individually when this cell is run.
report_date               = input("Report date (YYYY-MM-DD): ").strip()
grade_threshold           = float(input("Grade threshold (0–1, e.g. 0.85): ").strip())
missing_alert_threshold   = int(input("Action needed if missing assignments >= : ").strip())
late_alert_threshold      = int(input("Monitor if late assignments >= : ").strip())
low_score_alert_threshold = int(input("Action needed if low-score assignments >= : ").strip())

report_date = pd.Timestamp(report_date, tz='UTC')

print(
    f"\ndate={report_date.date()} | grade_threshold={grade_threshold} | "
    f"missing>={missing_alert_threshold}→action | late>={late_alert_threshold}→monitor | "
    f"low_score>={low_score_alert_threshold}→action"
)

print("\nProvide the path for each input file (e.g. /Users/you/Downloads/file.csv):")
SUMMARY_FILE = input("Canvas enrollment summary file path: ").strip()
CANVAS_FILE  = input("Canvas assignment report file path : ").strip()
COGNOS_FILE  = input("Cognos student data file path      : ").strip()

OUTPUT_FILE  = f"advisor_report_{report_date.strftime('%Y%m%d')}.xlsx"
print(f"\nOutput will be saved as: {OUTPUT_FILE}")

In [ ]:
# Cell 3 — Column Order Configuration
# To reorder: move an item to a different group, or move a whole group up/down.
# Assignment columns are always appended last, auto-sorted by due date.
STATIC_COL_GROUPS = [
    ["NAME", "ID", "MSU_EMAIL", "PERSONAL_EMAIL", "PHONE_NUMBER"],
    ["REGISTRATION_COHORT_DESCRIPTION", "ACADEMIC_STANDING", "CUMULATIVE_GPA", "PROGRAM_GPA"],
    ["LOW_GRADE_LIST", "CLASSES_REGISTERED_CURRENT_TERM", "CURRENT_COURSE_IN_PROGRESS"],
    ["current_score", "final_score", "current_grade", "final_grade"],
    ["total_assignments", "submitted_count", "missing_count", "late_count", "low_score_count"],
    ["alert_status"],
    # Assignment columns follow automatically — do not add them here
]

In [ ]:
# Cell 4 — Load Files

def load_file(path):
    return pd.read_excel(path) if path.lower().endswith(('.xlsx', '.xls')) else pd.read_csv(path)

df_summary = load_file(SUMMARY_FILE)
df_canvas  = load_file(CANVAS_FILE)
df_cognos  = load_file(COGNOS_FILE)

# Normalize column names for summary and canvas; leave cognos as-is (selected by exact name later)
df_summary.columns = df_summary.columns.str.strip().str.lower().str.replace(' ', '_')
df_canvas.columns  = df_canvas.columns.str.strip().str.lower()

print(f"Loaded — summary: {df_summary.shape}, canvas: {df_canvas.shape}, cognos: {df_cognos.shape}")

In [ ]:
# Cell 5 — Preprocess

# ── Summary ──────────────────────────────────────────────────────────────────
df_summary = df_summary[df_summary['enrollment_state'] == 'active'].copy()
df_summary['student_sis'] = df_summary['student_sis'].astype(str).str.strip()

# ── Canvas ───────────────────────────────────────────────────────────────────
df_canvas['sis_user_id'] = df_canvas['sis_user_id'].astype(str).str.strip()
df_canvas['assignment_due_at'] = pd.to_datetime(df_canvas['assignment_due_at'], utc=True, errors='coerce')

# Drop rows where due date could not be parsed
df_canvas = df_canvas[df_canvas['assignment_due_at'].notna()].copy()

# Include only assignments due on or before report_date
df_canvas = df_canvas[df_canvas['assignment_due_at'] <= report_date].copy()

# Drop assignments with a max possible score of 0 — placeholder/ungraded items
df_canvas = df_canvas[df_canvas['points_possible'] > 0].copy()

# Normalize late/missing to bool (Canvas exports these as TRUE/FALSE strings)
for col in ['late', 'missing']:
    df_canvas[col] = df_canvas[col].astype(str).str.upper().isin(['TRUE', '1', 'YES'])

# ── Cognos ───────────────────────────────────────────────────────────────────
# Strip M prefix from student ID to align with Canvas/Summary identifiers
df_cognos['ID'] = df_cognos['ID'].astype(str).str.strip().str.lstrip('M')

print(f"After filters — active enrollments: {len(df_summary)}, canvas rows: {len(df_canvas)}")

In [ ]:
# Cell 6 — Build Per-Course DataFrames

def get_assignment_status(missing, late):
    """missing takes precedence over late."""
    if missing:
        return 'missing'
    if late:
        return 'late'
    return 'good'


def sanitize_col(name, max_len=40):
    """Convert an assignment name to a safe DataFrame column prefix."""
    return (
        name.strip()
            .replace(' ', '_')
            .replace('/', '_')
            .replace('-', '_')
            .replace('(', '')
            .replace(')', '')
            [:max_len]
    )


def build_course_df(df_cv, df_sm):
    """
    Builds a wide-format DataFrame for a single course.
    Rows = students; columns = static fields + per-assignment status/grade pairs
    (sorted by due date, appended last).
    """
    # Sort assignments by due date
    assignments = (
        df_cv[['assignment_name', 'assignment_due_at']]
        .drop_duplicates('assignment_name')
        .sort_values('assignment_due_at')['assignment_name']
        .tolist()
    )

    records = []
    for _, srow in df_sm.iterrows():
        sid = srow['student_sis']
        sc  = df_cv[df_cv['sis_user_id'] == sid]

        missing_count   = 0
        late_count      = 0
        low_score_count = 0
        submitted_count = 0
        record = {'student_sis': sid}

        for asgn in assignments:
            arow = sc[sc['assignment_name'] == asgn]

            if arow.empty:
                # No canvas record for this student/assignment — treat as missing
                status = 'missing'
                grade  = np.nan
            else:
                arow   = arow.iloc[0]
                status = get_assignment_status(arow['missing'], arow['late'])
                grade  = round(arow['score'] / arow['points_possible'] * 100, 2)

            if status == 'missing':
                missing_count += 1
            else:
                submitted_count += 1
                if status == 'late':
                    late_count += 1
                # Count submitted assignments below grade threshold as low score
                if not np.isnan(grade) and (grade / 100) < grade_threshold:
                    low_score_count += 1

            col = sanitize_col(asgn)
            record[f'{col}_status'] = status
            record[f'{col}_grade']  = grade

        record['total_assignments'] = len(assignments)
        record['submitted_count']   = submitted_count
        record['missing_count']     = missing_count
        record['late_count']        = late_count
        record['low_score_count']   = low_score_count

        # Action Needed takes precedence over Monitor
        if missing_count >= missing_alert_threshold or low_score_count >= low_score_alert_threshold:
            record['alert_status'] = 'Action Needed'
        elif late_count >= late_alert_threshold:
            record['alert_status'] = 'Monitor'
        else:
            record['alert_status'] = 'Good'

        records.append(record)

    df_wide = pd.DataFrame(records)

    # Merge in course-level scores from summary
    score_cols = ['student_sis', 'current_score', 'final_score', 'current_grade', 'final_grade']
    available  = [c for c in score_cols if c in df_sm.columns]
    df_wide    = df_wide.merge(df_sm[available], on='student_sis', how='left')

    return df_wide, assignments


# ── Cognos columns carried into each sheet ───────────────────────────────────
COGNOS_KEEP = [
    'ID', 'NAME', 'MSU_EMAIL', 'PERSONAL_EMAIL', 'PHONE_NUMBER',
    'REGISTRATION_COHORT_DESCRIPTION', 'ACADEMIC_STANDING',
    'CUMULATIVE_GPA', 'PROGRAM_GPA',
    'LOW_GRADE_LIST', 'CLASSES_REGISTERED_CURRENT_TERM', 'CURRENT_COURSE_IN_PROGRESS',
]
df_cognos_slim = df_cognos[[c for c in COGNOS_KEEP if c in df_cognos.columns]].copy()


def apply_col_order(df, assignment_cols):
    """Order columns per STATIC_COL_GROUPS; assignment columns always last."""
    static = [c for group in STATIC_COL_GROUPS for c in group if c in df.columns]
    return df[[c for c in static + assignment_cols if c in df.columns]]


# ── Build one DataFrame per course ───────────────────────────────────────────
course_sheets = {}  # {descriptive_name: df}

for course_code in df_canvas['course_name'].unique():
    df_cv = df_canvas[df_canvas['course_name'] == course_code].copy()
    df_sm = df_summary[df_summary['course'] == course_code].copy()

    if df_sm.empty:
        continue

    # Descriptive name (e.g. 'DATA DRIVEN MARKETING') used for sheet tab and display
    descriptive_name = df_cv['course_section'].iloc[0]

    df_wide, assignments = build_course_df(df_cv, df_sm)

    # Build assignment column list: status then grade per assignment, in due-date order
    asgn_cols = [
        col
        for asgn in assignments
        for col in (f'{sanitize_col(asgn)}_status', f'{sanitize_col(asgn)}_grade')
    ]

    df_wide = df_wide.merge(df_cognos_slim, left_on='student_sis', right_on='ID', how='left')
    course_sheets[descriptive_name] = apply_col_order(df_wide, asgn_cols)

    print(f"  {descriptive_name}: {len(df_wide)} students, {len(assignments)} assignments")

In [ ]:
# Cell 7 — Build Summary Sheet
# One row per student; student info columns + per-course status/counts.

summary_parts = []

for course_name, df_course in course_sheets.items():
    id_col = 'ID' if 'ID' in df_course.columns else 'student_sis'
    keep   = [c for c in [id_col, 'NAME', 'alert_status', 'missing_count', 'late_count', 'low_score_count']
              if c in df_course.columns]
    temp   = df_course[keep].copy()

    # Use first 20 chars of course name as column prefix to keep headers readable
    prefix = course_name[:20].strip()
    temp   = temp.rename(columns={
        'alert_status':    f'{prefix}_status',
        'missing_count':   f'{prefix}_missing',
        'late_count':      f'{prefix}_late',
        'low_score_count': f'{prefix}_low_score',
    })
    summary_parts.append(temp)

df_summary_out = summary_parts[0]
id_col = 'ID' if 'ID' in df_summary_out.columns else 'student_sis'

for part in summary_parts[1:]:
    df_summary_out = df_summary_out.merge(part, on=[id_col, 'NAME'], how='outer')

# Overall status: worst-case across all courses
status_cols = [c for c in df_summary_out.columns if c.endswith('_status')]

def overall_status(row):
    vals = row[status_cols].tolist()
    if 'Action Needed' in vals:
        return 'Action Needed'
    if 'Monitor' in vals:
        return 'Monitor'
    return 'Good'

df_summary_out.insert(2, 'overall_status', df_summary_out.apply(overall_status, axis=1))

print(f"Summary sheet: {df_summary_out.shape}")

In [ ]:
# Cell 8 — Write Excel Output

def safe_sheet_name(name, used, max_len=31):
    """Truncate to Excel's 31-char sheet name limit and ensure uniqueness."""
    base = name[:max_len]
    if base not in used:
        return base
    for i in range(1, 100):
        candidate = f"{name[:max_len - 3]}_{i}"
        if candidate not in used:
            return candidate
    raise ValueError(f"Could not generate a unique sheet name for: {name}")


with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    # Summary sheet written first
    df_summary_out.to_excel(writer, sheet_name='Summary', index=False)

    used_names = {'Summary'}
    for course_name, df_course in course_sheets.items():
        sheet_name = safe_sheet_name(course_name, used_names)
        used_names.add(sheet_name)
        df_course.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Report written → {OUTPUT_FILE}")